In [25]:
#create yearly detections
import arcpy

gdb = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE.gdb"
master = f"{gdb}\\merged_94_22_nlcd_size_filtered_prebd_min_nonzero"

arcpy.env.overwriteOutput = True

for year in range(1994, 2023):

    out_fc = f"{gdb}\\SEFM_raw_{year}"

    if arcpy.Exists(out_fc):
        arcpy.management.Delete(out_fc)

    where = f"year = {year}"

    arcpy.analysis.Select(
        in_features=master,
        out_feature_class=out_fc,
        where_clause=where
    )

    print(f"Created {out_fc}")

Created C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE.gdb\SEFM_raw_1994
Created C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE.gdb\SEFM_raw_1995
Created C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE.gdb\SEFM_raw_1996
Created C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE.gdb\SEFM_raw_1997
Created C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE.gdb\SEFM_raw_1998
Created C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE.gdb\SEFM_raw_1999
Created C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE.gdb\SEFM_raw_2000
Created C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE.gdb\SEFM_raw_2001
Created C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE.gdb\SEFM_raw_2002
Created C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE.gdb\SEFM_raw_2003
Created C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE.gdb\SEFM_raw_2004
Created C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE.gdb\SEFM_raw_2005
Created C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE.gdb\SEFM_raw_2006
Created C:\U

In [27]:
#function for spatial/temporal clustering
def assign_events_for_year(year):

    import arcpy
    from datetime import datetime

    # ------------------------------------------------------------
    # Setup
    # ------------------------------------------------------------
    gdb = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE.gdb"
    fc = f"{gdb}\\SEFM_raw_{year}"              # raw polygons
    buffer_fc = f"{gdb}\\SEFM_90m_buffer_{year}" # 90 m buffer
    sj_fc = f"{gdb}\\sj_{year}"                  # spatial join output

    arcpy.env.overwriteOutput = True

    # Ensure event_ID field exists
    fields = [f.name for f in arcpy.ListFields(fc)]
    if "event_id" not in fields:
        arcpy.management.AddField(fc, "event_id", "TEXT", field_length=20)

    # ------------------------------------------------------------
    # 1. Create 90 m buffer for adjacency
    # ------------------------------------------------------------
    arcpy.analysis.Buffer(
        in_features=fc,
        out_feature_class=buffer_fc,
        buffer_distance_or_field="90 Meters"
    )

    # ------------------------------------------------------------
    # 2. Spatial join: buffer → raw polygons
    # ------------------------------------------------------------
    # Using INTERSECT because the buffer already encodes the 90 m distance.
    arcpy.analysis.SpatialJoin(
        target_features=buffer_fc,
        join_features=fc,
        out_feature_class=sj_fc,
        join_operation="JOIN_ONE_TO_MANY",
        match_option="INTERSECT"
    )

    # ------------------------------------------------------------
    # 3. Build adjacency list using detection_id
    # ------------------------------------------------------------
    adj = {}
    with arcpy.da.SearchCursor(sj_fc, ["detection_id", "detection_id_1"]) as cur:
        for tid, jid in cur:
            if tid == jid:
                continue
            adj.setdefault(tid, set()).add(jid)
            adj.setdefault(jid, set()).add(tid)

    # Ensure all detections appear in adjacency
    all_ids = [row[0] for row in arcpy.da.SearchCursor(fc, ["detection_id"])]
    for det in all_ids:
        adj.setdefault(det, set())

    # ------------------------------------------------------------
    # 4. Build spatial clusters (connected components)
    # ------------------------------------------------------------
    visited = set()
    spatial_clusters = []

    for det in all_ids:
        if det in visited:
            continue
        stack = [det]
        cluster = []
        while stack:
            x = stack.pop()
            if x in visited:
                continue
            visited.add(x)
            cluster.append(x)
            stack.extend(adj[x] - visited)
        spatial_clusters.append(cluster)

    # ------------------------------------------------------------
    # 5. Load temporal intervals
    # ------------------------------------------------------------
    intervals = {}
    with arcpy.da.SearchCursor(
        fc,
        ["detection_id", "prebd_min_corrected", "bd_min_corrected_plus8"]
    ) as cur:
        for det, start, end in cur:
            intervals[det] = (start, end)

    # ------------------------------------------------------------
    # 6. Temporal grouping within each spatial cluster
    # ------------------------------------------------------------
    event_lookup = {}
    multi_counter = 1
    single_counter = 1

    for cluster in spatial_clusters:

        # Single polygon cluster → singleton event
        if len(cluster) == 1:
            det = cluster[0]
            event_lookup[det] = f"S{year}_{single_counter:04d}"
            single_counter += 1
            continue

        # Sort by start time
        cluster_sorted = sorted(cluster, key=lambda x: intervals[x][0])

        current_group = []
        groups = []

        for det in cluster_sorted:
            start, end = intervals[det]

            if not current_group:
                current_group = [det]
                last_end = end
                continue

            if start <= last_end:
                current_group.append(det)
                last_end = max(last_end, end)
            else:
                groups.append(current_group)
                current_group = [det]
                last_end = end

        if current_group:
            groups.append(current_group)

        # Assign event IDs to temporal groups
        for g in groups:
            if len(g) == 1:
                det = g[0]
                event_lookup[det] = f"S{year}_{single_counter:04d}"
                single_counter += 1
            else:
                eid = f"{year}_{multi_counter:04d}"
                for det in g:
                    event_lookup[det] = eid
                multi_counter += 1

    # ------------------------------------------------------------
    # 7. Write event_ids back to the feature class
    # ------------------------------------------------------------
    with arcpy.da.UpdateCursor(fc, ["detection_id", "event_id"]) as cur:
        for row in cur:
            det = row[0]
            row[1] = event_lookup[det]
            cur.updateRow(row)

In [12]:
#test one year
assign_events_for_year(2020)

In [28]:
#seems ok, try all years
for year in range(1994, 2023):
    assign_events_for_year(year)

In [6]:
#check stats
import arcpy

gdb = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE.gdb"
arcpy.env.workspace = gdb

def summarize_events(year):
    fc = f"{gdb}\\SEFM_raw_{year}"

    if not arcpy.Exists(fc):
        print(f"{year}: MISSING")
        return None

    single_events = set()
    multi_events = set()
    single_detections = 0
    multi_detections = 0

    with arcpy.da.SearchCursor(fc, ["event_ID"]) as cur:
        for (eid,) in cur:
            if eid.startswith("S"):
                single_events.add(eid)
                single_detections += 1
            else:
                multi_events.add(eid)
                multi_detections += 1

    total = single_detections + multi_detections
    pct_multi = (multi_detections / total) * 100 if total > 0 else 0

    return {
        "year": year,
        "singleton_events": len(single_events),
        "multi_events": len(multi_events),
        "singleton_detections": single_detections,
        "multi_detections": multi_detections,
        "total_detections": total,
        "pct_multi": pct_multi
    }

# Print summary
print("YEAR | Total | Multi Det | Single Det | Multi Events | Single Events | % Multi")
print("-" * 90)

for year in range(1994, 2023):
    stats = summarize_events(year)
    if stats:
        print(f"{stats['year']} | "
              f"{stats['total_detections']:>5} | "
              f"{stats['multi_detections']:>10} | "
              f"{stats['singleton_detections']:>11} | "
              f"{stats['multi_events']:>12} | "
              f"{stats['singleton_events']:>13} | "
              f"{stats['pct_multi']:6.2f}")

YEAR | Total | Multi Det | Single Det | Multi Events | Single Events | % Multi
------------------------------------------------------------------------------------------
1994 | 43261 |      13894 |       29367 |         5269 |         29367 |  32.12
1995 | 46619 |      15364 |       31255 |         5784 |         31255 |  32.96
1996 | 69122 |      22627 |       46495 |         8554 |         46495 |  32.73
1997 | 51493 |      16607 |       34886 |         6200 |         34886 |  32.25
1998 | 57700 |      18477 |       39223 |         6938 |         39223 |  32.02
1999 | 110007 |      34074 |       75933 |        13015 |         75933 |  30.97
2000 | 118122 |      35543 |       82579 |        13269 |         82579 |  30.09
2001 | 101014 |      29305 |       71709 |        11075 |         71709 |  29.01
2002 | 95919 |      30308 |       65611 |        11247 |         65611 |  31.60
2003 | 69254 |      21675 |       47579 |         7971 |         47579 |  31.30
2004 | 98948 |      32375 |

In [5]:
#join event_id back to larger dataset
import arcpy

gdb = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE.gdb"
master = f"{gdb}\\merged_94_22_nlcd_size_filtered_prebd_min_nonzero"

# 1. Ensure event_id exists on master
fields = [f.name for f in arcpy.ListFields(master)]
if "event_ID" not in fields:
    arcpy.management.AddField(master, "event_ID", "TEXT", field_length=20)

# 2. Build global lookup: detection_id -> event_id
lookup = {}

for year in range(1994, 2023):
    raw_fc = f"{gdb}\\SEFM_raw_{year}"
    with arcpy.da.SearchCursor(raw_fc, ["detection_id", "event_ID"]) as cur:
        for det, eid in cur:
            if det is not None and eid is not None:
                lookup[det] = eid

print(f"Lookup size: {len(lookup)} detection_ids")

# 3. Update master using the lookup
with arcpy.da.UpdateCursor(master, ["detection_id", "event_ID"]) as cur:
    for det, eid in cur:
        if det in lookup:
            cur.updateRow([det, lookup[det]])

Lookup size: 2592194 detection_ids


In [7]:
#count detections per event
import arcpy
from collections import Counter

gdb = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE.gdb"
master = f"{gdb}\\merged_94_22_nlcd_size_filtered_prebd_min_nonzero"

# Count detections per event
event_sizes = Counter()

with arcpy.da.SearchCursor(master, ["event_id"]) as cur:
    for eid, in cur:
        if eid is not None:
            event_sizes[eid] += 1

In [8]:
#count single vs multi detection events
single_events = sum(1 for size in event_sizes.values() if size == 1)
multi_events  = sum(1 for size in event_sizes.values() if size > 1)

total_events = single_events + multi_events

In [9]:
#calculate percent multi vs single events
pct_single = 100 * single_events / total_events
pct_multi  = 100 * multi_events / total_events

print("Single‑detection events:", pct_single)
print("Multi‑detection events:", pct_multi)

Single‑detection events: 85.2403022300186
Multi‑detection events: 14.759697769981395


In [10]:
#filter to multideteciton events only
multi_sizes = [size for size in event_sizes.values() if size > 1]

In [11]:
#range
min_size = min(multi_sizes)
max_size = max(multi_sizes)

print("Minimum detections in a multi‑detection event:", min_size)
print("Maximum detections in a multi‑detection event:", max_size)

Minimum detections in a multi‑detection event: 2
Maximum detections in a multi‑detection event: 1048


In [12]:
from collections import Counter

hist = Counter(multi_sizes)

print("Histogram of detections per multi‑detection event:")
for size, count in sorted(hist.items()):
    print(f"{size} detections: {count} events")

Histogram of detections per multi‑detection event:
2 detections: 214810 events
3 detections: 52757 events
4 detections: 18086 events
5 detections: 7962 events
6 detections: 4072 events
7 detections: 2344 events
8 detections: 1473 events
9 detections: 944 events
10 detections: 690 events
11 detections: 492 events
12 detections: 326 events
13 detections: 260 events
14 detections: 192 events
15 detections: 162 events
16 detections: 124 events
17 detections: 105 events
18 detections: 79 events
19 detections: 80 events
20 detections: 56 events
21 detections: 51 events
22 detections: 54 events
23 detections: 43 events
24 detections: 23 events
25 detections: 34 events
26 detections: 21 events
27 detections: 18 events
28 detections: 24 events
29 detections: 21 events
30 detections: 13 events
31 detections: 16 events
32 detections: 11 events
33 detections: 14 events
34 detections: 15 events
35 detections: 6 events
36 detections: 11 events
37 detections: 9 events
38 detections: 5 events
39 detec

In [13]:
#look at event with 1048 detections
import arcpy
from collections import Counter

gdb = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE.gdb"
master = f"{gdb}\\merged_94_22_nlcd_size_filtered_prebd_min_nonzero"

# Count detections per event
event_sizes = Counter()
with arcpy.da.SearchCursor(master, ["event_id"]) as cur:
    for eid, in cur:
        if eid:
            event_sizes[eid] += 1

# Find the event with 1048 detections
big_event = [eid for eid, size in event_sizes.items() if size == 1048]
print(big_event)


['2009_11096']


In [14]:
#look at event with 918 detections
import arcpy
from collections import Counter

gdb = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE.gdb"
master = f"{gdb}\\merged_94_22_nlcd_size_filtered_prebd_min_nonzero"

# Count detections per event
event_sizes = Counter()
with arcpy.da.SearchCursor(master, ["event_id"]) as cur:
    for eid, in cur:
        if eid:
            event_sizes[eid] += 1

# Find the event with 1048 detections
big_event = [eid for eid, size in event_sizes.items() if size == 918]
print(big_event)


['2014_11197']


In [33]:
#look at event with 787 detections
import arcpy
from collections import Counter

gdb = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE.gdb"
master = f"{gdb}\\merged_94_22_nlcd_size_filtered_prebd_min_nonzero"

# Count detections per event
event_sizes = Counter()
with arcpy.da.SearchCursor(master, ["event_id"]) as cur:
    for eid, in cur:
        if eid:
            event_sizes[eid] += 1

# Find the event with 1048 detections
big_event = [eid for eid, size in event_sizes.items() if size == 787]
print(big_event)

['2017_13482']


In [40]:
#look at event with 65 detections
import arcpy
from collections import Counter

gdb = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE.gdb"
master = f"{gdb}\\merged_94_22_nlcd_size_filtered_prebd_min_nonzero"

# Count detections per event
event_sizes = Counter()
with arcpy.da.SearchCursor(master, ["event_id"]) as cur:
    for eid, in cur:
        if eid:
            event_sizes[eid] += 1

# Find the event with 1048 detections
big_event = [eid for eid, size in event_sizes.items() if size == 65]
print(big_event)

['2014_6958']


In [15]:
#i want average area and range of areas for detections in each multidetection event
import arcpy
from collections import defaultdict
import statistics
import csv

gdb = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE.gdb"
master = f"{gdb}\\merged_94_22_nlcd_size_filtered_prebd_min_nonzero"

# 1. Collect areas per event
areas = defaultdict(list)

with arcpy.da.SearchCursor(master, ["event_id", "area_ha"]) as cur:
    for eid, area in cur:
        if eid is not None and area is not None:
            areas[eid].append(area)

# 2. Compute stats for multi-detection events only
event_stats = []

for eid, vals in areas.items():
    if len(vals) > 1:  # multi-detection only
        event_stats.append({
            "event_id": eid,
            "n_detections": len(vals),
            "min_ha": min(vals),
            "max_ha": max(vals),
            "avg_ha": statistics.mean(vals)
        })

# 3. Write to CSV
csv_path = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\multi_detection_event_stats_09FEB2026.csv"

with open(csv_path, "w", newline="") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=["event_id", "n_detections", "min_ha", "max_ha", "avg_ha"]
    )
    writer.writeheader()
    writer.writerows(event_stats)

print("CSV export complete.")

CSV export complete.
